# Sentiment Model Comparison

Side-by-side evaluation of two sentiment classification approaches trained on the Flipkart product review dataset.

---

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load saved metrics
with open('../ml/sentiment/model/baseline_metrics.json') as f:
    baseline = json.load(f)

with open('../ml/sentiment/model/distilbert/metrics.json') as f:
    distilbert = json.load(f)

## 1. Performance Comparison Table

In [ ]:
comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'F1 (macro)', 'F1 (weighted)'],
    'TF-IDF + LogReg': [baseline['accuracy'], baseline['f1_macro'], baseline['f1_weighted']],
    'DistilBERT': [distilbert['accuracy'], distilbert['f1_macro'], distilbert['f1_weighted']],
})
comparison['Improvement'] = (
    (comparison['DistilBERT'] - comparison['TF-IDF + LogReg']) / comparison['TF-IDF + LogReg'] * 100
).round(1).astype(str) + '%'

print(comparison.to_string(index=False))

## 2. Visual Comparison

In [ ]:
metrics_names = ['Accuracy', 'F1 (macro)', 'F1 (weighted)']
baseline_vals = [baseline['accuracy'], baseline['f1_macro'], baseline['f1_weighted']]
distilbert_vals = [distilbert['accuracy'], distilbert['f1_macro'], distilbert['f1_weighted']]

x = np.arange(len(metrics_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, baseline_vals, width, label='TF-IDF + LogReg', color='#3498db', edgecolor='black')
bars2 = ax.bar(x + width/2, distilbert_vals, width, label='DistilBERT', color='#e74c3c', edgecolor='black')

# Add value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontweight='bold', fontsize=11)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontweight='bold', fontsize=11)

ax.set_ylabel('Score')
ax.set_title('Baseline vs DistilBERT — Sentiment Classification')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=12)
ax.axhline(y=0.87, color='gray', linestyle='--', alpha=0.5, label='Majority class baseline')

plt.tight_layout()
plt.savefig('../notebooks/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Trade-off Analysis

| Factor | TF-IDF + LogReg | DistilBERT |
|--------|----------------|------------|
| **Training time** | ~2 seconds | ~6.5 minutes |
| **Prediction speed** | <1ms per review | ~50ms per review |
| **Model size** | ~2 MB | ~267 MB |
| **F1 (macro)** | 0.4725 | 0.6285 |
| **Accuracy** | 88.76% | 93.26% |
| **Requires GPU?** | No | No (but benefits) |
| **Handles context?** | No (bag of words) | Yes (attention) |

## 4. Decision: Which Model for Production?

**We chose TF-IDF + LogReg for the real-time chatbot** because:
1. **Latency**: <1ms vs ~50ms — matters for real-time chat UX
2. **Memory**: 2MB vs 267MB — important for deployment
3. **Good enough**: For enriching chatbot responses (showing a sentiment badge), we don't need state-of-the-art accuracy

**DistilBERT is kept for**:
- Batch analysis of new reviews
- Offline evaluation and reporting
- Future improvement when GPU is available

## 5. Known Limitations

Both models struggle with:
- **Neutral class**: Only 26 training samples (5.9%) — insufficient for reliable detection
- **Sarcasm/irony**: "What a *great* product" (sarcastic) would be predicted as Positive
- **Domain shift**: Trained only on audio products — may not generalize to other categories

**Mitigation**: We use `class_weight='balanced'` and custom `WeightedTrainer` to partially address imbalance.